# QuantumMC Jupyter 入口

这个 Notebook 是用于在 Jupyter 中直接运行 Julia 版 MCWF 的入口模板。

## 1) 环境准备
首次运行前，请确保在仓库根目录执行过：`julia --project -e 'using Pkg; Pkg.instantiate()'`。

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()

In [ ]:
using QuantumMC
using QuantumMC.MCWFOptimized
using DataFrames
using CSV
using MAT

## 2) 单点运行（先做 sanity check）

In [ ]:
params = MCWFParams()
cfg = RunConfig(trajectories=20_000, i_sat=0.6, detuning=0.0, seed=20260406)
result = run_ensemble(params, cfg)
println("count = ", result.count)
println("sum(probability) = ", sum(result.probability))

## 3) 扫描示例（功率 × 失谐）

In [ ]:
powers = [0.15, 0.3, 0.45, 0.6, 0.75, 0.9]
detunings = collect(-1.0:0.1:1.0)
rows = DataFrame(i_sat=Float64[], detuning=Float64[], count=Float64[], prob_sum=Float64[])

for pwr in powers, Δ in detunings
    cfg = RunConfig(trajectories=100_000, i_sat=pwr, detuning=Δ, seed=20260406)
    r = run_ensemble(params, cfg)
    push!(rows, (pwr, Δ, r.count, sum(r.probability)))
end

first(rows, 6)

## 4) 导出结果（CSV + MAT）

In [ ]:
mkpath("../output")
CSV.write("../output/scan_summary.csv", rows)

matwrite("../output/scan_summary.mat", Dict(
    "i_sat" => rows.i_sat,
    "detuning" => rows.detuning,
    "count" => rows.count,
    "prob_sum" => rows.prob_sum
))

## 5) 与 MATLAB 参考结果对比（可选）
若有 `matlab_reference.mat`（包含 `probability_ref`），可调用仓库脚本做自动校验。

In [ ]:
# 在 notebook 内直接运行脚本
run(`julia --project=.. ../scripts/verify_against_matlab.jl ../matlab_reference.mat`)